# Data re-uploading
**Quantum Machine Learning · Week 3 (Variational quantum classifiers) · Day 4**

Lesson page: https://sara-aminpour.com/tutorials/qml/w3d4.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

**Data re-uploading.** A circuit that loads $x$ through rotation gates computes a Fourier series in $x$. The encoding
decides which frequencies exist; the trainable gates only choose their amplitudes. Every extra upload adds a frequency,
and that's why Monday's classifier got stuck on the moons.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator

estimator = StatevectorEstimator()

def one_qubit_model(L):
    """A trainable rotation W_0, then L times: upload x with RX(x), then another trainable W_l = RZ·RY·RZ."""
    x, w = ParameterVector("x", 1), ParameterVector("w", 3 * (L + 1))
    qc = QuantumCircuit(1)
    for l in range(L + 1):
        if l > 0:
            qc.rx(x[0], 0)                          # upload number l
        qc.rz(w[3 * l], 0)
        qc.ry(w[3 * l + 1], 0)
        qc.rz(w[3 * l + 2], 0)
    return qc

def evaluate(circuit, xs, weight_sets, observable="Z"):
    """⟨observable⟩ for every x and every weight vector: shape (weight sets, points)."""
    rows = np.array([np.concatenate([W, np.atleast_1d(x)]) for W in weight_sets for x in xs])
    evs = estimator.run([(circuit, SparsePauliOp(observable), rows)]).result()[0].data.evs
    return evs.reshape(len(weight_sets), len(xs))

one_qubit_model(3).draw("mpl", style="clifford")

## Count the frequencies
Sample the model on 64 points of $[-\pi, \pi)$ with random weights and take the Fourier transform.

In [ ]:
def spectrum(values):
    """Amplitudes of the frequencies 0 to 5 of a real function sampled on 64 evenly spaced points."""
    c = np.fft.fft(values) / len(values)
    return np.r_[np.abs(c[0]), 2 * np.abs(c[1:6])]

grid = np.linspace(-np.pi, np.pi, 64, endpoint=False)
rng = np.random.default_rng(0)
for L in (1, 2, 3):
    model = one_qubit_model(L)
    for trial in range(2):
        W = rng.uniform(0, 2 * np.pi, size=model.num_parameters - 1)
        print(f"L = {L}, random weights {trial + 1}: amplitudes of frequencies 0..5 =", spectrum(evaluate(model, grid, [W])[0]).round(3))

## Fit a target that needs frequency 3
$g(x) = 0.45\sin x + 0.3\cos 2x - 0.25\sin 3x$, from 24 samples, with parameter-shift gradients and Adam.

In [ ]:
target = lambda x: 0.45 * np.sin(x) + 0.3 * np.cos(2 * x) - 0.25 * np.sin(3 * x)
xs = np.linspace(-np.pi, np.pi, 24, endpoint=False)

def fit(L, steps=150, lr=0.1, seed=0):
    model = one_qubit_model(L)
    W = np.random.default_rng(seed).uniform(0, 2 * np.pi, size=model.num_parameters - 1)
    m, v = np.zeros_like(W), np.zeros_like(W)
    shifts = [W * 0 + s * np.eye(len(W))[j] for j in range(len(W)) for s in (np.pi / 2, -np.pi / 2)]
    for k in range(1, steps + 1):
        V = evaluate(model, xs, [W] + [W + d for d in shifts])
        f, df = V[0], (V[1::2] - V[2::2]) / 2
        g = df @ (2 * (f - target(xs))) / len(xs)
        m, v = 0.9 * m + 0.1 * g, 0.999 * v + 0.001 * g ** 2                  # Adam, as in PyTorch
        W = W - lr * (m / (1 - 0.9 ** k)) / (np.sqrt(v / (1 - 0.999 ** k)) + 1e-8)
    return model, W, np.mean((evaluate(model, xs, [W])[0] - target(xs)) ** 2)

fits = {L: fit(L) for L in (1, 2, 3)}
for L, (model, W, err) in fits.items():
    print(f"L = {L}: {model.num_parameters - 1:2d} trainable angles, final error {err:.4f}")

import matplotlib.pyplot as plt
fine = np.linspace(-np.pi, np.pi, 241)
fig, axes = plt.subplots(2, 3, figsize=(11, 5.4), layout="constrained", gridspec_kw={"height_ratios": [1.35, 1]})
for col, (L, (model, W, err)) in enumerate(fits.items()):
    ax = axes[0, col]
    ax.plot(fine, target(fine), "--", color="#8a85a3", lw=2, label="target")
    ax.plot(fine, evaluate(model, fine, [W])[0], color="#ff7a59", lw=2.4, label="model")
    ax.scatter(xs, target(xs), color="#8a85a3", s=14, zorder=3)
    ax.set_title(f"{L} upload{'s' if L > 1 else ''}: error {err:.4f}")
    ax.set_xticks([-np.pi, 0, np.pi], [r"$-\pi$", "0", r"$\pi$"])
    ax.set_ylim(-1.15, 1.15)
    bars = axes[1, col]
    freq = np.arange(6)
    bars.bar(freq - 0.18, spectrum(target(grid)), width=0.34, color="#bdb8cf", label="target")
    bars.bar(freq + 0.18, spectrum(evaluate(model, grid, [W])[0]), width=0.34, color="#ff7a59", label="model")
    bars.axvspan(L + 0.5, 5.6, color="#e5566b", alpha=0.08)
    bars.set_xticks(freq)
    bars.set_xlabel("frequency")
    bars.set_ylim(0, 0.55)
axes[0, 0].legend(frameon=False, loc="lower left")
axes[1, 0].set_ylabel("amplitude")
plt.show()

## Back to the moons
Monday's classifier uploaded each point once. Upload it in every layer instead: same circuit size, same 12 weights.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

def classifier(layers=3, reupload=True):
    x, w = ParameterVector("x", 2), ParameterVector("w", 4 * layers)
    qc = QuantumCircuit(2)
    for l in range(layers):
        if reupload or l == 0:                    # Monday: only before the first layer
            qc.ry(x[0], 0)
            qc.ry(x[1], 1)
        qc.ry(w[4 * l], 0)
        qc.rz(w[4 * l + 1], 0)
        qc.ry(w[4 * l + 2], 1)
        qc.rz(w[4 * l + 3], 1)
        qc.cx(0, 1)
    return qc

X, y = make_moons(n_samples=120, noise=0.1, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=40, random_state=0, stratify=y)
scaler = MinMaxScaler((-np.pi / 2, np.pi / 2)).fit(X_train)
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)

def train(circuit, epochs=60, lr=0.1, seed=0):
    t = 1 - 2 * y_train
    W = 0.6 * np.random.default_rng(seed).normal(size=circuit.num_parameters - 2)
    m, v = np.zeros_like(W), np.zeros_like(W)
    shifts = [s * np.eye(len(W))[j] for j in range(len(W)) for s in (np.pi / 2, -np.pi / 2)]
    for k in range(1, epochs + 1):
        V = evaluate(circuit, X_train, [W] + [W + d for d in shifts], "ZI")
        f, df = V[0], (V[1::2] - V[2::2]) / 2
        g = df @ (2 * (f - t)) / len(t)
        m, v = 0.9 * m + 0.1 * g, 0.999 * v + 0.001 * g ** 2
        W = W - lr * (m / (1 - 0.9 ** k)) / (np.sqrt(v / (1 - 0.999 ** k)) + 1e-8)
    return W, np.mean((f - t) ** 2)

accuracy = lambda c, W, X_, y_: np.mean((evaluate(c, X_, [W], "ZI")[0] < 0).astype(int) == y_)
results = {}
for name, reup in (("uploaded once", False), ("re-uploaded", True)):
    c = classifier(3, reup)
    W, loss = train(c)
    results[name] = (c, W)
    print(f"{name:14s}: loss {loss:.3f}, train accuracy {accuracy(c, W, X_train, y_train):.3f}, test accuracy {accuracy(c, W, X_test, y_test):.3f}")

from matplotlib.colors import LinearSegmentedColormap
apricot_blue = LinearSegmentedColormap.from_list("apricot_blue", ["#5577f5", "#ffffff", "#ff8a50"])
g = np.linspace(-2.05, 2.05, 45)
gx, gy = np.meshgrid(g, g)
fig, axes = plt.subplots(1, 2, figsize=(9, 4.2), layout="constrained")
for ax, (name, (c, W)) in zip(axes, results.items()):
    s = evaluate(c, np.c_[gx.ravel(), gy.ravel()], [W], "ZI")[0].reshape(gx.shape)
    ax.contourf(gx, gy, s, levels=np.linspace(-1, 1, 21), cmap=apricot_blue, alpha=0.6)
    ax.contour(gx, gy, s, levels=[0], colors="#2c2842", linewidths=2)
    ax.scatter(X_train[:, 0], X_train[:, 1], c=np.where(y_train == 0, "#ff8a50", "#5577f5"), s=20, edgecolors="white", linewidths=0.6)
    ax.set_aspect("equal")
    ax.set_title(f"3 layers, {name}")
plt.show()

## Your turn: upload in parallel
Sequential uploads on one qubit are one route to higher frequencies. The other: put the same $x$ on several qubits at
once. Build the 3-qubit model below and compute its spectrum with random weights, with and without the CNOTs.

In [ ]:
def parallel_model(n=3, entangle=True):
    x, w = ParameterVector("x", 1), ParameterVector("w", 3 * n)
    qc = QuantumCircuit(n)
    for q in range(n):
        qc.ry(w[q], q)
        qc.rz(w[n + q], q)
    for q in range(n):
        qc.rx(x[0], q)                            # the same x on every qubit, all at once
    if entangle:
        for q in range(n - 1, 0, -1):
            qc.cx(q, q - 1)                       # gather the qubits' information onto qubit 0
    for q in range(n):
        qc.ry(w[2 * n + q], q)
    return qc

# TODO: for entangle in (True, False): evaluate ⟨Z⟩ of qubit 0 (observable "IIZ") on the 64-point grid
#       with random weights, and print spectrum(...). Which frequencies appear?

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
rng = np.random.default_rng(1)
for entangle in (True, False):
    model = parallel_model(3, entangle)
    W = rng.uniform(0, 2 * np.pi, size=model.num_parameters - 1)
    amps = spectrum(evaluate(model, grid, [W], "IIZ")[0])
    print(f"{'with' if entangle else 'without'} CNOTs: amplitudes of frequencies 0..5 =", amps.round(3))
print("Three parallel uploads reach frequency 3, but only if the CNOTs carry the other qubits' information")
print("to the measured qubit; without them, qubit 0 only ever sees its own single upload.")